# Lab 2a · Quanto pão assar amanhã
**Oficina do Builder · Winovoito** — Bancada 2 · ML aplicado

A rede fictícia **Pão da Esquina** tem cinco lojas e três anos de vendas diárias. Sobra pão todo dia, e às vezes falta. Seu trabalho: prever a venda de amanhã, por loja, e mostrar em reais quanto isso economiza.

Neste lab você:
1. gera os dados sintéticos e explora os padrões;
2. separa treino, validação e teste **no tempo**;
3. cria o baseline e anota o número a vencer;
4. cria características de calendário e defasagens, **sem olhar para o futuro**;
5. treina um gradient boosting dentro de um pipeline e compara com o baseline;
6. traduz o erro em pão que vai para o lixo.

> Os dados são sintéticos, gerados com padrões conhecidos. Nenhuma chave de API é necessária neste lab.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)

## 1 · Os dados

In [ ]:
from oficina.sinteticos import gerar_padaria
from oficina import demanda

df = gerar_padaria(semente=7)
print(df.shape)
df.head()

## 2 · Explorar
Cada loja tem um ritmo próprio. Olhe o dia da semana: a Praia e a Estação contam histórias opostas.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(df.set_index("data").groupby("loja")["vendas"].resample("W").sum().unstack(0)
   .plot(ax=ax[0], title="Vendas semanais por loja"))
(df.assign(dia_semana=df["data"].dt.day_name().str[:3])
   .pivot_table(index="dia_semana", columns="loja", values="vendas", aggfunc="mean")
   .reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
   .plot(kind="bar", ax=ax[1], title="Média por dia da semana"))
plt.tight_layout()

## 3 · Dividir no tempo
Treino até o mês 30, validação 31 a 33, teste 34 a 36. **Nunca sorteie linhas** quando os dados têm data: no mundo real você prevê o futuro com o passado.

In [ ]:
feat = demanda.criar_caracteristicas(df).dropna(subset=demanda.CARACTERISTICAS)
treino, valid, teste = demanda.dividir_no_tempo(feat, inicio="2023-01-01")
for nome, parte in [("treino", treino), ("validação", valid), ("teste", teste)]:
    print(f"{nome:10s} {parte['data'].min().date()} a {parte['data'].max().date()}  ({len(parte)} linhas)")

## 4 · O baseline: o número a vencer
"Amanhã vende o mesmo que no mesmo dia da semana passada." Anote o MAE dele.

> **Cuidado com o MAPE:** ele divide pelo valor real. Em dias de venda perto de zero, o MAPE explode e deixa de dizer alguma coisa. Para conversar com o negócio, o MAE, em pães, é mais honesto.

In [ ]:
base_valid = demanda.baseline_sazonal(valid)
print(f"Baseline na validação: MAE = {demanda.mae(valid['vendas'], base_valid):.1f} pães/dia · "
      f"MAPE = {demanda.mape(valid['vendas'], base_valid):.1f}%")

## 5 · Características que só olham para o passado
Abra `oficina/demanda.py` e leia `criar_caracteristicas`. A regra de ouro: **`shift` antes de `rolling`**. A linha do dia *t* só pode usar vendas até *t−1*.

### Exercício 1 · Crie uma característica nova
Escreva a sua versão abaixo, acrescentando uma característica que você acha útil (por exemplo, a venda do mesmo dia no ano anterior). Depois rode o teste de vazamento.

In [ ]:
def minhas_caracteristicas(d):
    f = demanda.criar_caracteristicas(d)
    g = f.groupby("loja")["vendas"]
    # TODO: sua característica nova. Exemplo seguro: venda de 364 dias atrás (mesmo dia da semana)
    f["lag_364"] = g.shift(364)
    return f

MINHAS = demanda.CARACTERISTICAS + ["lag_364"]

### O teste de vazamento
A função troca por lixo todas as vendas a partir de uma data *t* e recalcula. Se alguma característica do dia *t* mudar, ela olhou para o presente ou para o futuro.

In [ ]:
vazamentos = demanda.checar_vazamento(df, minhas_caracteristicas, colunas=MINHAS)
print("✅ Sem vazamento" if not vazamentos else f"❌ Vazamento em: {vazamentos}")

# Veja o detector funcionando: uma média móvel SEM shift
def com_vazamento(d):
    f = demanda.criar_caracteristicas(d)
    f["media_7"] = f.groupby("loja")["vendas"].transform(lambda s: s.rolling(7).mean())
    return f
print("Média móvel sem shift →", demanda.checar_vazamento(df, com_vazamento))

## 6 · Pipeline + gradient boosting, com validação cruzada no tempo
O pipeline garante que tudo o que aprende com os dados aprenda só com o treino. A `TimeSeriesSplit` treina até um ponto e testa logo depois, várias vezes.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

feat2 = minhas_caracteristicas(df).dropna(subset=MINHAS)
treino, valid, teste = demanda.dividir_no_tempo(feat2, inicio="2023-01-01")
treino_ord = treino.sort_values("data")

pipe = demanda.criar_pipeline()
cv = TimeSeriesSplit(n_splits=4)
maes = -cross_val_score(pipe, treino_ord[MINHAS], treino_ord["vendas"], cv=cv,
                        scoring="neg_mean_absolute_error")
print("MAE por rodada:", maes.round(1), "· média:", maes.mean().round(1))

pipe.fit(treino[MINHAS], treino["vendas"])
print(f"Validação: modelo {demanda.mae(valid['vendas'], pipe.predict(valid[MINHAS])):.1f} × "
      f"baseline {demanda.mae(valid['vendas'], demanda.baseline_sazonal(valid)):.1f}")

## 7 · A avaliação final, no teste
Só agora, com tudo decidido, você olha o teste. Treine com treino + validação e meça uma vez.

In [ ]:
final = pd.concat([treino, valid])
pipe = demanda.criar_pipeline().fit(final[MINHAS], final["vendas"])
teste = teste.assign(previsto=pipe.predict(teste[MINHAS]), baseline=demanda.baseline_sazonal(teste))
mae_m, mae_b = demanda.mae(teste["vendas"], teste["previsto"]), demanda.mae(teste["vendas"], teste["baseline"])
print(f"MAE no teste: modelo {mae_m:.1f} × baseline {mae_b:.1f} → {1 - mae_m / mae_b:.0%} menor")
print("Critério do lab: pelo menos 15% abaixo do baseline →", "✅" if mae_m <= 0.85 * mae_b else "❌")

### Onde o modelo erra mais?
Quase sempre a resposta ensina mais do que a média.

In [ ]:
erro = teste.assign(erro_abs=(teste["vendas"] - teste["previsto"]).abs(),
                   dia=teste["data"].dt.day_name().str[:3])
display(erro.groupby("loja")["erro_abs"].mean().round(1).to_frame("MAE por loja"))
display(erro.pivot_table(index="dia", values="erro_abs", aggfunc="mean")
            .reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]).round(1))

## 8 · Do erro ao negócio
Hipótese do lab: a loja produz exatamente o previsto. O que passa da venda vira sobra (custo fictício de R$ 0,60 por pão).

In [ ]:
com_base = demanda.sobra_semanal(teste["vendas"], teste["baseline"])
com_modelo = demanda.sobra_semanal(teste["vendas"], teste["previsto"])
pd.DataFrame({"Baseline": com_base, "Modelo": com_modelo})

## 9 · Reflexão
Responda em uma ou duas frases cada:
1. O que acontece com o modelo numa **loja nova**, sem histórico?
2. E se o preço do pão mudar 20%?
3. Que evento poderia quebrar todos os padrões de uma vez (lembre de 2020)?
4. Este problema pedia mesmo ML? Volte à Ficha de decisão da Bancada 1 e justifique.

> **Revisor dos 7 erros:** cole este notebook e o Revisor (pasta `revisor/`) numa conversa nova com o seu assistente de IA e leia os achados.